**Part 3: Policy RAG | Step 3: Retrieval evaluation**  
Goal: measure how often the correct clause is found, before any LLM is used.  
`any@k` = at least one correct clause is in the top k. `all@k` = every correct clause is in the top k.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd

# Find the project root: the folder that contains src/policy_rag.py
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "policy_rag.py").exists())
sys.path.append(str(ROOT / "src"))
import policy_rag as rag

POLICY_DIR = ROOT / "notebooks" / "RAG" / "policy"
OUT_DIR = ROOT / "notebooks" / "RAG" / "outputs"
OUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

**Step 1. Load the 20 evaluation questions.** 16 have an answer in the policy and 4 do not.

**Where the evaluation questions came from**

The 20 evaluation questions are not from the brief or from any official set. I wrote them myself, with help from AI, by reading the policy.

For each question we also chose the correct clause IDs (`gold_ids`) and a short expected answer. Everything is saved in `policy/eval_questions.csv`.

**What I did to make sure they are reliable**
- All 20 questions were written from the policy text, with AI help.
- I read every question and checked that its `gold_ids` is really the clause (or clauses) that answers it.
- 16 questions have an answer in the policy. The other 4 are on purpose not covered, to test that the system says "not found".

In [2]:
eval_df = pd.read_csv(POLICY_DIR / "eval_questions.csv")
print(eval_df["answerable"].value_counts().rename({1: "answerable", 0: "unanswerable"}).to_string())
eval_df[["qid", "question", "gold_ids", "answerable"]]

answerable
answerable      16
unanswerable     4


,qid,question,gold_ids,answerable
0,Q01,A seller has 96% on-time delivery and an 11% adverse review rate. ...,MOP-1.2;MOP-1.3,1
1,Q02,How many completed orders does a seller need in the 90-day window ...,MOP-1.1,1
2,Q03,What commission rate does a Gold seller pay?,MOP-1.5,1
3,Q04,An order arrives 7 days after the estimated delivery date. What is...,MOP-2.2,1
4,Q05,A standard-freight order to the Northeast arrives 6 days late. How...,MOP-2.4;MOP-2.2,1
5,Q06,An Express Freight order to the Northeast arrives 3 days late. How...,MOP-8.4;MOP-2.4,1
6,Q07,An order worth R$ 500 is delivered 12 days late. How much platform...,MOP-3.2;MOP-2.2,1
7,Q08,"Who has to approve a compensation of R$ 1,500?",MOP-3.3,1
8,Q09,When can a customer get a cash refund instead of platform credit?,MOP-4.1;MOP-3.1,1
9,Q10,How long does a refund take if the customer paid by Pix?,MOP-4.2,1


**Step 2. Compare search settings.** Same questions, four settings. Title = section title added to the embedded text.

**Title and plain, in the simplest way**

Think of the model as a reader. Before it makes the numbers for a clause, we decide **what text it reads**.

- **plain** = it reads only the clause.
- **title** = it reads the section name first, then the clause.

**Example with clause 6.1**

**plain** reads this:

```
6.1 A seller entering Watch tier is placed in a 30-day Improvement Period and assigned an operations contact. Listings remain active.
```

**title** reads this:

```
Seller Escalation and Suspension. 6.1 A seller entering Watch tier is placed in a 30-day Improvement Period and assigned an operations contact. Listings remain active.
```

The only difference is the extra words "Seller Escalation and Suspension" at the front. These words tell the reader the topic of the clause.

**Why we try both**

We do not know in advance which one gives better search. So we build both and test them on the same 16 questions.

- `idx_plain` = the 33 clause vectors made from the **plain** text
- `idx_title` = the 33 clause vectors made from the **title + clause** text

Everything else is the same: same model, same questions, same dense search. So any difference in the score comes from the title alone.

**What the result says**

- dense + title: the right clause was ranked first for 16 of 16 questions.
- dense, no title (plain): 15 of 16.

So the title helped on 1 question. It is a small gain on a small test.

**Short version**

- **plain** = clause only
- **title** = section name + clause
- We built both to see if adding the topic name helps the search find the right clause. Here it helped a little.

In [3]:
chunks = rag.chunk_policy(rag.load_policy(POLICY_DIR / "marketplace_policy.txt"))
idx_title = rag.PolicyIndex(chunks, with_title=True)
idx_plain = rag.PolicyIndex(chunks, with_title=False)

configs = {
    "dense + title":   (idx_title, "dense"),
    "dense, no title": (idx_plain, "dense"),
    "bm25":            (idx_title, "bm25"),
    "hybrid + title":  (idx_title, "hybrid"),
}
summ = {}
for name, (idx, mode) in configs.items():
    per_q, s = rag.evaluate_retrieval(idx, eval_df, ks=(1, 3, 5), mode=mode)
    summ[name] = s
pd.DataFrame(summ).T.round(2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,any@1,all@1,any@3,all@3,any@5,all@5,MRR
dense + title,1.00,0.56,1.00,0.94,1.0,1.00,1.00
"dense, no title",0.94,0.56,1.00,0.94,1.0,1.00,0.97
bm25,0.75,0.44,0.94,0.88,1.0,0.94,0.86
hybrid + title,0.81,0.50,1.00,0.94,1.0,1.00,0.91


**Reading the output**

There are 16 answerable questions, so every score is a fraction of 16. For example, 0.94 = 15 of 16, and 0.81 = 13 of 16.

The scores of the four settings (any@1, all@1, any@3, all@3, any@5, all@5, MRR):
- **dense + title:** 1.00, 0.56, 1.00, 0.94, 1.0, 1.00, 1.00
- **dense, no title:** 0.94, 0.56, 1.00, 0.94, 1.0, 1.00, 0.97
- **bm25:** 0.75, 0.44, 0.94, 0.88, 1.0, 0.94, 0.86
- **hybrid + title:** 0.81, 0.50, 1.00, 0.94, 1.0, 1.00, 0.91

**What the columns mean**

The number after the @ is k, the count of top results we look at.
- **any@k:** at least one correct clause is in the top k.
- **all@k:** every correct clause is in the top k.
- **MRR:** how high the first correct clause is ranked, averaged over questions. Rank 1 gives 1.0, rank 2 gives 0.5, rank 3 gives 0.33.

**Example**

Take Q01. Its correct clauses are `MOP-1.2` and `MOP-1.3`. Say the search returns `MOP-1.2`, `MOP-1.3`, `MOP-1.1`.
- any@1 is yes, because the top 1 is `MOP-1.2`.
- all@1 is no, because one result cannot hold both clauses.
- any@3 and all@3 are both yes.

**Why all@1 is never above 0.56**

all@1 looks at only the top 1 result. It asks: is every correct clause inside that 1 result?

Some questions have one correct clause, and some have two.

One correct clause (9 questions):
- Q02 needs `MOP-1.1`
- Q03 needs `MOP-1.5`
- Q04 needs `MOP-2.2`
- Q08 needs `MOP-3.3`
- Q10 needs `MOP-4.2`
- Q12 needs `MOP-5.4`
- Q14 needs `MOP-6.4`
- Q15 needs `MOP-7.1`
- Q16 needs `MOP-8.3`

Two correct clauses (7 questions):
- Q01 needs `MOP-1.2` and `MOP-1.3`
- Q05 needs `MOP-2.4` and `MOP-2.2`
- Q06 needs `MOP-8.4` and `MOP-2.4`
- Q07 needs `MOP-3.2` and `MOP-2.2`
- Q09 needs `MOP-4.1` and `MOP-3.1`
- Q11 needs `MOP-5.2` and `MOP-2.3`
- Q13 needs `MOP-6.2` and `MOP-6.1`

With k=1 we keep only one clause. For a two-clause question like Q01, even a perfect search returns only `MOP-1.2`, and `MOP-1.3` is still missing. So these 7 questions can never pass all@1, for any search.

Only the 9 one-clause questions can pass. The best possible all@1 is 9 out of 16 = **0.56**.

Dense + title shows 0.56. It passed all 9 questions that were possible, so 0.56 is a perfect score here. It is not a weak score.

A simple way to remember: all@1 is like carrying 2 bags with only 1 hand. If a question needs 2 bags, it can never pass, however strong the person is.

For a fair comparison of the settings, look at any@1 and at all@3 or all@5. Use all@1 only as a rough check, with 0.56 as the maximum.

**Row by row**

- **dense + title:** perfect on any@1 (16 of 16) and MRR (1.00). The correct clause is always ranked first. At k=3 one question still misses a clause (all@3 is 0.94). At k=5 everything is found.
- **dense, no title:** one question (15 of 16) does not have a correct clause at rank 1. The MRR of 0.97 fits that question having its correct clause at rank 2. Everything else is the same as with the title, so the title gave a small gain.
- **bm25:** the weakest. It has a correct clause at rank 1 for only 12 of 16 questions. Even at k=5 it misses a clause for one question (all@5 is 0.94). Searching by words alone is not enough.
- **hybrid + title:** 13 of 16 at rank 1, so worse than dense at the top. The MRR of 0.91 fits the 3 misses being at rank 2. At k=3 and k=5 it ties with dense + title. The likely reason is that BM25 is the weaker list, and hybrid gives it equal weight.

**What to take away**

- **Rank 1:** dense + title is the best setting.
- **k=3:** one question still misses a clause. For hybrid it is Q07, where `MOP-2.2` is missing (shown in Step 3 of the notebook).
- **k=5:** dense + title and hybrid find every correct clause (all@5 is 1.00).
- **Caution:** this is 16 questions that were written from the policy, with policy words in them. A difference of 1 to 3 questions is a hint, not proof.

**Step 3. Questions that still fail with the hybrid setting at k=3.** Shows the correct clauses next to what was retrieved.

In [4]:
per_q, _ = rag.evaluate_retrieval(idx_title, eval_df, ks=(1, 3, 5), mode="hybrid")
gold = eval_df.set_index("qid")["gold_ids"]
miss = per_q[~per_q["all@3"]].copy()
miss["gold"] = miss["qid"].map(gold)
miss["top3"] = miss["retrieved"].str[:3].str.join(", ")
miss[["qid", "gold", "top3", "any@3", "all@3"]]

,qid,gold,top3,any@3,all@3
6,Q07,MOP-3.2;MOP-2.2,"MOP-3.2, MOP-3.1, MOP-3.4",True,False



```
Q07   gold: MOP-3.2;MOP-2.2   top3: MOP-3.2, MOP-3.1, MOP-3.4
```

- **gold** is what the search should find: `MOP-3.2` and `MOP-2.2`.
- **top3** is what the search found: `MOP-3.2`, `MOP-3.1`, `MOP-3.4`.
- `MOP-3.2` is in both lists, so that clause was found.
- `MOP-2.2` is in gold but not in top3, so that clause was missed.

That is all the output says: one clause out of two was found.

**Why Q07 needs two clauses**

The question is: "An order worth R$ 500 is delivered 12 days late. How much credit does the customer get?"

To answer it you need two facts:
- **Fact A, in `MOP-2.2`:** a delay of 11 to 20 days is called Severe. This tells you that 12 days means Severe.
- **Fact B, in `MOP-3.2`:** for a Severe delay, the credit is the greater of `R$ 50 or 20%` of the order value. So the credit is R$ 100.

Without A, you do not know that 12 days is Severe. Without B, you do not know how much a Severe delay pays. The answer needs both.

**What went wrong**

The search found Fact B (`MOP-3.2`) but missed Fact A (`MOP-2.2`). The question has words like "credit", and the clauses `MOP-3.1` and `MOP-3.4` also talk about credit and compensation, so they probably ranked above `MOP-2.2`. The question never uses the word "Severe", so nothing in it points to `MOP-2.2`.

**Why it matters**: If the next step gives only these three clauses to the LLM, it sees the rule for Severe delays but not the rule that says 12 days is Severe. It may say "not found", or guess. Both are wrong.

**The fix**: Look at more results. With the top 5 (k=5), `MOP-2.2` is found. That is why the `all@5` column is 1.00, and why I suggested using k=5 in notebook 04.

**Short version**: Q07 is the only question where the search found one of its two needed clauses and missed the other. Taking 5 results instead of 3 fixes it.

**In Depth Question Level Analysis**

**first_rank and k_needed**

These are two numbers that tell you where the correct clauses landed in the search results.

Both are measured in ranks. Rank 1 means the best result, rank 2 the second best, and so on.

- **first_rank:** the rank of the first correct clause that appears.
- **k_needed:** the rank of the last correct clause. This is the smallest k (number of results) you need to take to get all the correct clauses.

**Example with Q07**

The correct clauses are `MOP-3.2` and `MOP-2.2`. Suppose the search ranks the clauses like this:
- rank 1: `MOP-3.2` (correct)
- rank 2: `MOP-7.1`
- rank 3: `MOP-3.1`
- rank 4: `MOP-2.2` (correct)

Then:
- `first_rank` = 1, because the first correct clause is at rank 1.
- `k_needed` = 4, because the last correct clause is at rank 4. You must take the top 4 to get both.

That is why Q07 fails at k=3 and works at k=5.

**Example with a one-clause question**

For Q02, the only correct clause is `MOP-1.1`. If it is at rank 1, then `first_rank` = 1 and `k_needed` = 1. With one correct clause, the two numbers are always equal.

**What each one is used for**

- `first_rank` gives MRR: 1 divided by `first_rank`, averaged over questions. It tells you if the best clause is on top.
- `k_needed` tells you how many results to send to the LLM. If the highest `k_needed` across all questions is 4, then k=4 is enough to give the LLM every needed clause for every question.

**How to read it in your table**

- `first_rank` is 1 and `k_needed` is 1: perfect.
- `first_rank` is 1 and `k_needed` is above 1: the question needs several clauses and some rank lower. Check that `k_needed` is not above the k you use.
- A big `k_needed` such as 9 or 20 means the setting buried a needed clause.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

ids = [c["chunk_id"] for c in chunks]

# A "ranker" is a function: question -> all 33 clause IDs, best first.
# One ranker for each (method, variant).
rankers = {}
for variant, idx, with_title in [("plain", idx_plain, False), ("plain+header", idx_title, True)]:
    texts = [rag.embed_text(c, with_title) for c in chunks]    # plain = clause only, header = title + clause

    # TF-IDF: rows are normalised, so the dot product is the cosine similarity
    tfidf = TfidfVectorizer(lowercase=True)
    mat = tfidf.fit_transform(texts)

    def tfidf_ranker(q, tfidf=tfidf, mat=mat):
        scores = (mat @ tfidf.transform([q]).T).toarray().ravel()
        return [ids[i] for i in np.argsort(-scores, kind="stable")]
    rankers[("TF-IDF", variant)] = tfidf_ranker

    # BM25, dense and hybrid come from the index we already built
    for method, mode in [("BM25", "bm25"), ("Dense (cosine)", "dense"), ("Hybrid", "hybrid")]:
        rankers[(method, variant)] = (
            lambda q, idx=idx, mode=mode: [h["chunk_id"] for h in idx.search(q, k=len(ids), mode=mode)])


def question_table(ranker, eval_df):
    """One row per answerable question, with every metric."""
    rows = []
    for r in eval_df[eval_df["answerable"] == 1].itertuples():
        ranked = ranker(r.question)
        gold_list = rag.parse_gold(r.gold_ids)
        gold = set(gold_list)
        row = {"qid": r.qid, "gold": ", ".join(gold_list),
               "top1": ranked[0], "top3": ", ".join(ranked[:3]), "top5": ", ".join(ranked[:5])}
        for k in (1, 3, 5):
            row[f"any@{k}"] = bool(gold & set(ranked[:k]))     # at least one correct clause in top k
            row[f"all@{k}"] = gold <= set(ranked[:k])          # every correct clause in top k
        row["first_rank"] = min(ranked.index(g) + 1 for g in gold)   # rank of the first correct clause
        row["k_needed"] = max(ranked.index(g) + 1 for g in gold)     # smallest k that finds ALL correct clauses
        rows.append(row)
    return pd.DataFrame(rows)


tables = {key: question_table(fn, eval_df) for key, fn in rankers.items()}
METHODS = ["TF-IDF", "BM25", "Dense (cosine)", "Hybrid"]
VARIANTS = ["plain", "plain+header"]

for method in METHODS:
    for variant in VARIANTS:
        print(f"\n=== {method} | {variant} ===")
        display(tables[(method, variant)])


=== TF-IDF | plain ===


,qid,gold,top1,top3,top5,any@1,all@1,any@3,all@3,any@5,all@5,first_rank,k_needed
0,Q01,"MOP-1.2, MOP-1.3",MOP-1.3,"MOP-1.3, MOP-1.2, MOP-5.2","MOP-1.3, MOP-1.2, MOP-5.2, MOP-1.4, MOP-1.5",True,False,True,True,True,True,1,2
1,Q02,MOP-1.1,MOP-1.1,"MOP-1.1, MOP-2.4, MOP-6.3","MOP-1.1, MOP-2.4, MOP-6.3, MOP-6.1, MOP-8.4",True,True,True,True,True,True,1,1
2,Q03,MOP-1.5,MOP-1.5,"MOP-1.5, MOP-1.2, MOP-1.3","MOP-1.5, MOP-1.2, MOP-1.3, MOP-3.4, MOP-7.1",True,True,True,True,True,True,1,1
3,Q04,MOP-2.2,MOP-2.1,"MOP-2.1, MOP-2.3, MOP-7.1","MOP-2.1, MOP-2.3, MOP-7.1, MOP-2.2, MOP-4.1",False,False,False,False,True,True,4,4
4,Q05,"MOP-2.4, MOP-2.2",MOP-2.2,"MOP-2.2, MOP-3.2, MOP-4.1","MOP-2.2, MOP-3.2, MOP-4.1, MOP-2.4, MOP-8.4",True,False,True,False,True,True,1,4
5,Q06,"MOP-8.4, MOP-2.4",MOP-2.2,"MOP-2.2, MOP-2.4, MOP-8.4","MOP-2.2, MOP-2.4, MOP-8.4, MOP-4.1, MOP-3.2",False,False,True,True,True,True,2,3
6,Q07,"MOP-3.2, MOP-2.2",MOP-4.1,"MOP-4.1, MOP-3.2, MOP-3.4","MOP-4.1, MOP-3.2, MOP-3.4, MOP-2.2, MOP-3.1",False,False,True,False,True,True,2,4
7,Q08,MOP-3.3,MOP-4.1,"MOP-4.1, MOP-6.2, MOP-3.3","MOP-4.1, MOP-6.2, MOP-3.3, MOP-3.1, MOP-3.2",False,False,True,True,True,True,3,3
8,Q09,"MOP-4.1, MOP-3.1",MOP-4.1,"MOP-4.1, MOP-3.1, MOP-4.3","MOP-4.1, MOP-3.1, MOP-4.3, MOP-4.2, MOP-6.4",True,False,True,True,True,True,1,2
9,Q10,MOP-4.2,MOP-8.4,"MOP-8.4, MOP-4.1, MOP-8.3","MOP-8.4, MOP-4.1, MOP-8.3, MOP-1.4, MOP-3.4",False,False,False,False,False,False,8,8



=== TF-IDF | plain+header ===


,qid,gold,top1,top3,top5,any@1,all@1,any@3,all@3,any@5,all@5,first_rank,k_needed
0,Q01,"MOP-1.2, MOP-1.3",MOP-1.2,"MOP-1.2, MOP-1.3, MOP-5.2","MOP-1.2, MOP-1.3, MOP-5.2, MOP-1.5, MOP-1.4",True,False,True,True,True,True,1,2
1,Q02,MOP-1.1,MOP-1.1,"MOP-1.1, MOP-6.3, MOP-2.4","MOP-1.1, MOP-6.3, MOP-2.4, MOP-6.1, MOP-1.4",True,True,True,True,True,True,1,1
2,Q03,MOP-1.5,MOP-1.5,"MOP-1.5, MOP-1.2, MOP-1.3","MOP-1.5, MOP-1.2, MOP-1.3, MOP-3.4, MOP-7.1",True,True,True,True,True,True,1,1
3,Q04,MOP-2.2,MOP-2.1,"MOP-2.1, MOP-2.3, MOP-2.2","MOP-2.1, MOP-2.3, MOP-2.2, MOP-7.1, MOP-4.1",False,False,True,True,True,True,3,3
4,Q05,"MOP-2.4, MOP-2.2",MOP-2.2,"MOP-2.2, MOP-3.2, MOP-4.1","MOP-2.2, MOP-3.2, MOP-4.1, MOP-2.4, MOP-8.4",True,False,True,False,True,True,1,4
5,Q06,"MOP-8.4, MOP-2.4",MOP-8.4,"MOP-8.4, MOP-2.2, MOP-2.4","MOP-8.4, MOP-2.2, MOP-2.4, MOP-4.1, MOP-8.3",True,False,True,True,True,True,1,3
6,Q07,"MOP-3.2, MOP-2.2",MOP-4.1,"MOP-4.1, MOP-3.2, MOP-3.4","MOP-4.1, MOP-3.2, MOP-3.4, MOP-3.1, MOP-2.2",False,False,True,False,True,True,2,5
7,Q08,MOP-3.3,MOP-4.1,"MOP-4.1, MOP-3.1, MOP-6.2","MOP-4.1, MOP-3.1, MOP-6.2, MOP-3.3, MOP-3.2",False,False,False,False,True,True,4,4
8,Q09,"MOP-4.1, MOP-3.1",MOP-4.3,"MOP-4.3, MOP-4.1, MOP-3.1","MOP-4.3, MOP-4.1, MOP-3.1, MOP-4.2, MOP-3.2",False,False,True,True,True,True,2,3
9,Q10,MOP-4.2,MOP-8.4,"MOP-8.4, MOP-4.1, MOP-4.2","MOP-8.4, MOP-4.1, MOP-4.2, MOP-8.3, MOP-3.4",False,False,True,True,True,True,3,3



=== BM25 | plain ===


...

,qid,gold,top1,top3,top5,any@1,all@1,any@3,all@3,any@5,all@5,first_rank,k_needed
0,Q01,"MOP-1.2, MOP-1.3",MOP-1.2,"MOP-1.2, MOP-1.3, MOP-1.5","MOP-1.2, MOP-1.3, MOP-1.5, MOP-1.1, MOP-1.4",True,False,True,True,True,True,1,2
1,Q02,MOP-1.1,MOP-1.1,"MOP-1.1, MOP-6.1, MOP-6.2","MOP-1.1, MOP-6.1, MOP-6.2, MOP-1.3, MOP-1.5",True,True,True,True,True,True,1,1
2,Q03,MOP-1.5,MOP-1.5,"MOP-1.5, MOP-1.3, MOP-7.1","MOP-1.5, MOP-1.3, MOP-7.1, MOP-1.2, MOP-3.3",True,True,True,True,True,True,1,1
3,Q04,MOP-2.2,MOP-2.2,"MOP-2.2, MOP-3.2, MOP-2.1","MOP-2.2, MOP-3.2, MOP-2.1, MOP-2.3, MOP-2.4",True,True,True,True,True,True,1,1
4,Q05,"MOP-2.4, MOP-2.2",MOP-2.4,"MOP-2.4, MOP-2.2, MOP-3.2","MOP-2.4, MOP-2.2, MOP-3.2, MOP-2.1, MOP-8.4",True,False,True,True,True,True,1,2
5,Q06,"MOP-8.4, MOP-2.4",MOP-2.4,"MOP-2.4, MOP-2.2, MOP-8.4","MOP-2.4, MOP-2.2, MOP-8.4, MOP-2.1, MOP-2.3",True,False,True,True,True,True,1,3
6,Q07,"MOP-3.2, MOP-2.2",MOP-3.2,"MOP-3.2, MOP-7.1, MOP-3.1","MOP-3.2, MOP-7.1, MOP-3.1, MOP-2.2, MOP-2.4",True,False,True,False,True,True,1,4
7,Q08,MOP-3.3,MOP-3.3,"MOP-3.3, MOP-3.1, MOP-3.2","MOP-3.3, MOP-3.1, MOP-3.2, MOP-7.2, MOP-3.4",True,True,True,True,True,True,1,1
8,Q09,"MOP-4.1, MOP-3.1",MOP-3.1,"MOP-3.1, MOP-4.3, MOP-4.1","MOP-3.1, MOP-4.3, MOP-4.1, MOP-4.2, MOP-7.1",True,False,True,True,True,True,1,3
9,Q10,MOP-4.2,MOP-4.2,"MOP-4.2, MOP-7.1, MOP-3.1","MOP-4.2, MOP-7.1, MOP-3.1, MOP-6.5, MOP-4.1",True,True,True,True,True,True,1,1



=== Hybrid | plain ===


,qid,gold,top1,top3,top5,any@1,all@1,any@3,all@3,any@5,all@5,first_rank,k_needed
0,Q01,"MOP-1.2, MOP-1.3",MOP-1.2,"MOP-1.2, MOP-1.3, MOP-5.2","MOP-1.2, MOP-1.3, MOP-5.2, MOP-1.4, MOP-1.1",True,False,True,True,True,True,1,2
1,Q02,MOP-1.1,MOP-1.1,"MOP-1.1, MOP-6.1, MOP-6.3","MOP-1.1, MOP-6.1, MOP-6.3, MOP-6.2, MOP-1.4",True,True,True,True,True,True,1,1
2,Q03,MOP-1.5,MOP-1.5,"MOP-1.5, MOP-1.3, MOP-1.2","MOP-1.5, MOP-1.3, MOP-1.2, MOP-7.1, MOP-6.1",True,True,True,True,True,True,1,1
3,Q04,MOP-2.2,MOP-2.1,"MOP-2.1, MOP-3.2, MOP-2.2","MOP-2.1, MOP-3.2, MOP-2.2, MOP-7.1, MOP-2.3",False,False,True,True,True,True,3,3
4,Q05,"MOP-2.4, MOP-2.2",MOP-3.2,"MOP-3.2, MOP-2.2, MOP-2.4","MOP-3.2, MOP-2.2, MOP-2.4, MOP-8.4, MOP-2.1",False,False,True,True,True,True,2,3
5,Q06,"MOP-8.4, MOP-2.4",MOP-2.4,"MOP-2.4, MOP-2.2, MOP-8.4","MOP-2.4, MOP-2.2, MOP-8.4, MOP-3.2, MOP-8.3",True,False,True,True,True,True,1,3
6,Q07,"MOP-3.2, MOP-2.2",MOP-3.2,"MOP-3.2, MOP-3.1, MOP-3.4","MOP-3.2, MOP-3.1, MOP-3.4, MOP-2.4, MOP-4.3",True,False,True,False,True,False,1,7
7,Q08,MOP-3.3,MOP-3.3,"MOP-3.3, MOP-3.2, MOP-3.1","MOP-3.3, MOP-3.2, MOP-3.1, MOP-3.4, MOP-4.1",True,True,True,True,True,True,1,1
8,Q09,"MOP-4.1, MOP-3.1",MOP-4.3,"MOP-4.3, MOP-4.1, MOP-3.1","MOP-4.3, MOP-4.1, MOP-3.1, MOP-4.2, MOP-2.3",False,False,True,True,True,True,2,3
9,Q10,MOP-4.2,MOP-4.1,"MOP-4.1, MOP-4.2, MOP-8.4","MOP-4.1, MOP-4.2, MOP-8.4, MOP-6.4, MOP-1.4",False,False,True,True,True,True,2,2



=== Hybrid | plain+header ===


,qid,gold,top1,top3,top5,any@1,all@1,any@3,all@3,any@5,all@5,first_rank,k_needed
0,Q01,"MOP-1.2, MOP-1.3",MOP-1.2,"MOP-1.2, MOP-1.3, MOP-1.5","MOP-1.2, MOP-1.3, MOP-1.5, MOP-1.4, MOP-6.1",True,False,True,True,True,True,1,2
1,Q02,MOP-1.1,MOP-1.1,"MOP-1.1, MOP-6.1, MOP-6.3","MOP-1.1, MOP-6.1, MOP-6.3, MOP-6.2, MOP-1.4",True,True,True,True,True,True,1,1
2,Q03,MOP-1.5,MOP-1.5,"MOP-1.5, MOP-1.3, MOP-1.2","MOP-1.5, MOP-1.3, MOP-1.2, MOP-7.1, MOP-6.1",True,True,True,True,True,True,1,1
3,Q04,MOP-2.2,MOP-2.1,"MOP-2.1, MOP-2.2, MOP-2.3","MOP-2.1, MOP-2.2, MOP-2.3, MOP-3.2, MOP-2.4",False,False,True,True,True,True,2,2
4,Q05,"MOP-2.4, MOP-2.2",MOP-3.2,"MOP-3.2, MOP-2.4, MOP-2.2","MOP-3.2, MOP-2.4, MOP-2.2, MOP-2.1, MOP-8.4",False,False,True,True,True,True,2,3
5,Q06,"MOP-8.4, MOP-2.4",MOP-2.4,"MOP-2.4, MOP-2.2, MOP-8.4","MOP-2.4, MOP-2.2, MOP-8.4, MOP-8.3, MOP-3.2",True,False,True,True,True,True,1,3
6,Q07,"MOP-3.2, MOP-2.2",MOP-3.2,"MOP-3.2, MOP-3.1, MOP-3.4","MOP-3.2, MOP-3.1, MOP-3.4, MOP-2.4, MOP-2.2",True,False,True,False,True,True,1,5
7,Q08,MOP-3.3,MOP-3.3,"MOP-3.3, MOP-3.2, MOP-3.1","MOP-3.3, MOP-3.2, MOP-3.1, MOP-4.1, MOP-3.4",True,True,True,True,True,True,1,1
8,Q09,"MOP-4.1, MOP-3.1",MOP-4.3,"MOP-4.3, MOP-3.1, MOP-4.1","MOP-4.3, MOP-3.1, MOP-4.1, MOP-4.2, MOP-3.2",False,False,True,True,True,True,2,3
9,Q10,MOP-4.2,MOP-4.2,"MOP-4.2, MOP-4.1, MOP-3.1","MOP-4.2, MOP-4.1, MOP-3.1, MOP-8.4, MOP-4.3",True,True,True,True,True,True,1,1


In [6]:
rows = []
for (method, variant), t in tables.items():
    rows.append({"method": method, "variant": variant,
                 **{c: round(t[c].mean(), 2) for c in ["any@1", "all@1", "any@3", "all@3", "any@5", "all@5"]},
                 "MRR": round((1 / t["first_rank"]).mean(), 2),     # over the full ranking
                 "k_needed_max": int(t["k_needed"].max())})         # k that finds everything for every question
summary = pd.DataFrame(rows)

# Rank the settings: all@5 first, then all@3, then MRR, then any@1
leaderboard = summary.sort_values(["all@5", "all@3", "MRR", "any@1"], ascending=False).reset_index(drop=True)
leaderboard.index = leaderboard.index + 1
leaderboard.index.name = "rank"
leaderboard

,method,variant,any@1,all@1,any@3,all@3,any@5,all@5,MRR,k_needed_max
rank,,,,,,,,,,
1,Dense (cosine),plain+header,1.00,0.56,1.00,0.94,1.00,1.00,1.00,4
2,Dense (cosine),plain,0.94,0.56,1.00,0.94,1.00,1.00,0.97,4
3,Hybrid,plain+header,0.81,0.50,1.00,0.94,1.00,1.00,0.91,5
4,TF-IDF,plain+header,0.69,0.38,0.94,0.81,1.00,1.00,0.81,5
5,Hybrid,plain,0.75,0.44,1.00,0.94,1.00,0.94,0.86,7
6,BM25,plain+header,0.75,0.44,0.94,0.88,1.00,0.94,0.86,11
7,TF-IDF,plain,0.62,0.31,0.88,0.75,0.94,0.94,0.76,8
8,BM25,plain,0.81,0.44,0.88,0.75,0.94,0.88,0.87,11


In [7]:
needed = pd.DataFrame({f"{m} | {v}": t.set_index("qid")["k_needed"] for (m, v), t in tables.items()})
display(needed)
print("questions that need k > 3, per setting:")
(needed > 3).sum()

,TF-IDF | plain,BM25 | plain,Dense (cosine) | plain,Hybrid | plain,TF-IDF | plain+header,BM25 | plain+header,Dense (cosine) | plain+header,Hybrid | plain+header
qid,,,,,,,,
Q01,2,2,2,2,2,2,2,2
Q02,1,1,1,1,1,1,1,1
Q03,1,1,1,1,1,1,1,1
Q04,4,6,1,3,3,5,1,2
Q05,4,4,2,3,4,3,2,3
Q06,3,2,3,3,3,2,3,3
Q07,4,11,4,7,5,11,4,5
Q08,3,1,1,1,4,1,1,1
Q09,2,3,3,3,3,3,3,3


questions that need k > 3, per setting:


TF-IDF | plain                   4
BM25 | plain                     4
Dense (cosine) | plain           1
Hybrid | plain                   1
TF-IDF | plain+header            3
BM25 | plain+header              2
Dense (cosine) | plain+header    1
Hybrid | plain+header            1
dtype: int64